<a href="https://colab.research.google.com/github/MatteoOnger/lama-lab/blob/main/notebooks/sweeps_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market-making sweeps on a Colab GPU runtime

Runs `delta_sweep.py` and `variance_sweep.py` (Sections 3.1 / 4 of the manuscript) using `AgentBlumMansourExp3` -- a vectorized Blum-Mansour implementation, bit-identical to `AgentBlumMansour` + a constant-eta `AgentExp3` expert, but computed as a handful of batched tensor ops instead of a Python loop over `n_arms` expert objects. That matters here specifically because it batches far better on a GPU than the per-expert-object loop did.

**Use the T4 GPU runtime, not a TPU.** This is a sequential online-learning loop -- each round depends on the previous round's weights, so there is nothing to parallelize across rounds, only within one (across episodes). PyTorch/XLA (the TPU backend) works by lazily tracing ops into a graph and compiling it, which is a poor match for millions of small sequential steps with per-step host-device syncs; it can end up *slower* than plain CPU. T4 (CUDA) has no such tracing tax, so it is the safer accelerator here -- `Runtime > Change runtime type > T4 GPU` before running anything below.

In [ ]:
# Do NOT run this cell in a local environment -- it's intended for Google Colab only.

# Clone GitHub repository
!git clone https://github.com/MatteoOnger/lama-lab.git

# Set working directory
%cd /content/lama-lab

# Install dependencies (CUDA-enabled torch is already present on the Colab GPU image)
%pip install -q -e .

# Set working directory
%cd /content/lama-lab/notebooks

## Confirm the GPU is actually there

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print(
        "No GPU detected -- check Runtime > Change runtime type > T4 GPU, "
        "then Runtime > Restart session. The scripts below fall back to CPU "
        "automatically, but that defeats the point of running this here."
    )

## Why these particular flags

Both scripts auto-detect CUDA (`device = "cuda" if torch.cuda.is_available() else "cpu"`), so no code changes are needed to use the GPU. One flag does need to change from the CPU-oriented defaults, though:

- **`--jobs 1`**. The `--jobs` flag spawns separate OS processes to use multiple *CPU* cores in parallel -- on a single GPU runtime, multiple processes would instead fight over one CUDA context, which helps nothing. Run everything in one process and let the GPU's own batched throughput do the work.

`--eta` is a **plain fixed learning rate** applied to every one of the `n_arms` Exp3 experts inside Blum-Mansour, not the horizon/arm-count formula you might expect (`sqrt(2 ln K / (T K))`) -- that formula is the right one for a single Exp3 expert's own worst-case *external*-regret bound, and turns out to be far too conservative for this swap-regret construction: at n_arms > ~30 it barely moves the policy off uniform within any affordable T. See `delta_sweep.py`'s module docstring for the full account (short version: Blum-Mansour's swap regret is O(n_arms^1.5 sqrt(T log n_arms)), so a non-vacuous bound needs T = Omega(n_arms^3) -- e.g. ~9.3 million rounds at n_arms=210, not the tens of thousands a CPU run can afford in reasonable time). `--eta 0.05` is what CPU testing validated as producing real, visible convergence at small n_arms; treat it as a starting point to watch, not a settled value, especially at the larger n_arms a GPU makes newly affordable.

**Watch the first `delta`/`std` point's timing before committing to the full sweep.** GPU throughput for this specific shape (small-ish batched matrices, a python-level round loop) is not something we've been able to benchmark locally -- if it's dramatically faster than CPU, push `--rounds` well past the defaults below; if kernel-launch overhead dominates and it's not much faster, scale back rather than let a run continue for hours to no benefit.

## Delta sweep (Theorem 4.8: envelope narrows as the tick size shrinks)

In [ ]:
# Starting point, not a final answer -- see the markdown above. Widen --deltas toward
# finer grids (e.g. add 0.1, 0.0667, 0.05) and/or raise --rounds once you've seen how
# fast this GPU actually runs one point.
!python ../scripts/delta_sweep.py \
    --deltas 0.5 0.3333333333333333 0.25 0.2 \
    --episodes 30 30 30 30 \
    --rounds 200000 \
    --eta 0.05 \
    --jobs 1 \
    --results_dir ../results

## Variance sweep (Proposition 3.10: spread scales linearly with sigma)

In [ ]:
# --delta here needs to stay coordinated with whatever n_arms range delta_sweep.py
# above showed actually converges -- a grid too coarse to resolve the smallest
# theoretical spread in --stds is a different failure mode than one too fine to
# converge in the round budget (see variance_sweep.py's module docstring).
!python ../scripts/variance_sweep.py \
    --stds 0.15 0.2 0.25 0.3 \
    --delta 0.2 \
    --episodes 30 \
    --rounds 200000 \
    --eta 0.05 \
    --jobs 1 \
    --results_dir ../results

## Save results

Download everything as a zip -- the Colab runtime is ephemeral.

In [ ]:
# Do NOT run this cell in a local environment -- it's intended for Google Colab only.

import shutil
from google.colab import files

# Compress the results folder into a ZIP archive
shutil.make_archive('../results', 'zip', '../results')

# Download the ZIP archive
files.download('../results.zip')